# Colab — End-to-End Fine-Tune & Predict
**Multilingual Health QA (HASH / Zindi).** Reproducible run: install → train seq2seq →
evaluate on Val (leaderboard-mirroring ROUGE) → generate a **reviewable** Test prediction
CSV. Maps to rubric crit. 1, 2, 9 (leaderboard, tracking, reproducibility).

> ⚠️ This notebook does **not** upload anything. It writes a local submission CSV for you
> to review and submit manually.

## 0. Runtime check (use a GPU runtime: Runtime → Change runtime type → T4/A100)

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1. Get the code + data

In [ ]:
import sys, os
if 'google.colab' in sys.modules:
    !git clone -q https://github.com/<your-user>/multilingual_health_qa.git
    %cd multilingual_health_qa
    !pip install -q -r requirements.txt
sys.path.insert(0, os.getcwd())
# Data must be in ./datasets (Train.csv, Val.csv, Test.csv, SampleSubmission.csv).
# On Colab, upload them or mount Drive if not tracked in the repo.
assert os.path.exists('datasets/Train.csv'), 'place data CSVs in datasets/'

## 2. Train
Model ladder by VRAM: `google/mt5-small` (fast) → `google/mt5-base` (default) →
`facebook/nllb-200-distilled-600M` (best African coverage). One variable at a time — log
each run in `experiments/`.

In [ ]:
# ~2-4h for mt5-base/3 epochs on a T4. Drop to mt5-small or fewer epochs to iterate fast.
!python -m src.train \
    --model_name google/mt5-base \
    --output_dir outputs/mt5base_v1 \
    --epochs 3 --train_bs 8 --eval_bs 16 --grad_accum 2 \
    --lr 3e-4 --max_input_len 128 --max_target_len 256 \
    --num_beams 4 --gen_max_len 256 --no_repeat_ngram 3 \
    --eval_subsample 1500 --fp16

## 3. Evaluate on Val (full per-language ROUGE)

In [ ]:
!python -m src.infer --model_dir outputs/mt5base_v1 --split val \
    --num_beams 4 --gen_max_len 256 --no_repeat_ngram 3 --tag mt5base_v1

## 4. Generate Test predictions → reviewable submission CSV (no upload)

In [ ]:
!python -m src.infer --model_dir outputs/mt5base_v1 --split test \
    --num_beams 4 --gen_max_len 256 --no_repeat_ngram 3 --tag mt5base_v1

In [ ]:
import pandas as pd
sub = pd.read_csv('outputs/submission_mt5base_v1.csv')
print(sub.shape); sub.head()  # REVIEW before uploading to Zindi

## 5. Manual review checklist (do this before submitting)
- [ ] Spot-check Amharic rows are **Ge'ez script** (infer.py prints a wrong-language count).
- [ ] No empty / single-token answers; lengths look reasonable per language.
- [ ] Val proxy **beats the TF-IDF B0 (0.394)** and prior runs (`experiments/results.csv`).
- [ ] IDs match `SampleSubmission.csv` (submit.py validates this).
Then upload `outputs/submission_*.csv` to Zindi yourself and record the LB score in the log.

## 6. Log the run
Append a row to `experiments/results.csv` and a note to `experiments/log.md`:
config (model, epochs, lr, decoding), Val R1/RL/proxy, LB score, and one-line takeaway.